In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('diabetic_data.csv')
# STEP 1: LOAD AND MAP HIDDEN MISSING VALUES

In [ ]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [ ]:
df.tail()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
101761,443847548,100162476,AfricanAmerican,Male,[70-80),?,1,3,7,3,...,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782,74694222,AfricanAmerican,Female,[80-90),?,1,4,5,5,...,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148,41088789,Caucasian,Male,[70-80),?,1,1,7,1,...,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166,31693671,Caucasian,Female,[80-90),?,2,3,7,10,...,No,Up,No,No,No,No,No,Ch,Yes,NO
101765,443867222,175429310,Caucasian,Male,[70-80),?,1,1,7,6,...,No,No,No,No,No,No,No,No,No,NO


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [ ]:
df.isnull().sum()

,0
encounter_id,0
patient_nbr,0
race,0
gender,0
age,0
weight,0
admission_type_id,0
discharge_disposition_id,0
admission_source_id,0
time_in_hospital,0


In [ ]:
df.replace('?', np.nan, inplace=True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      99493 non-null   object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    3197 non-null    object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                61510 non-null   object
 11  medical_specialty         51817 non-null   object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [ ]:
df.isnull().sum()

,0
encounter_id,0
patient_nbr,0
race,2273
gender,0
age,0
weight,98569
admission_type_id,0
discharge_disposition_id,0
admission_source_id,0
time_in_hospital,0


In [ ]:
df.isnull().sum()/df.shape[0]*100

,0
encounter_id,0.000000
patient_nbr,0.000000
race,2.233555
gender,0.000000
age,0.000000
weight,96.858479
admission_type_id,0.000000
discharge_disposition_id,0.000000
admission_source_id,0.000000
time_in_hospital,0.000000


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# 1. Define your features (X) and your target label (y)
X = df.drop(columns=['readmitted'])
y = df['readmitted'].map({'NO': 0, '>30': 1, '<30': 1}) # Binary mapping

# 2. Initialize the group splitter
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# 3. Get the split indices based on the patient groups
train_idx, test_idx = next(gss.split(X, y, groups=df['patient_nbr']))

# 4. Generate the final 4 distinct datasets
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [ ]:
# Find the most common value (mode) for race in the training set
train_race_mode = X_train['race'].mode()[0]

# Use .loc to cleanly fill missing values without triggering warnings
X_train.loc[:, 'race'] = X_train['race'].fillna(train_race_mode)
X_test.loc[:, 'race'] = X_test['race'].fillna(train_race_mode)

# Verify the result
print("Missing values in X_train['race'] after imputation:", X_train['race'].isnull().sum())

Missing values in X_train['race'] after imputation: 0


In [ ]:
# Selects columns where the data type is 'object' (text)
text_columns = X_train.select_dtypes(include=['object']).columns.tolist()

print("Columns with text values:")
print(text_columns)

Columns with text values:
['race', 'gender', 'age', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


In [ ]:
# Loop through the text columns and count their unique values
for col in X_train.select_dtypes(include=['object']).columns:
    print(f"{col}: {X_train[col].nunique()} unique text values")

race: 5 unique text values
gender: 3 unique text values
age: 10 unique text values
weight: 9 unique text values
payer_code: 16 unique text values
medical_specialty: 71 unique text values
diag_1: 700 unique text values
diag_2: 719 unique text values
diag_3: 754 unique text values
max_glu_serum: 3 unique text values
A1Cresult: 3 unique text values
metformin: 4 unique text values
repaglinide: 4 unique text values
nateglinide: 4 unique text values
chlorpropamide: 4 unique text values
glimepiride: 4 unique text values
acetohexamide: 1 unique text values
glipizide: 4 unique text values
glyburide: 4 unique text values
tolbutamide: 2 unique text values
pioglitazone: 4 unique text values
rosiglitazone: 4 unique text values
acarbose: 4 unique text values
miglitol: 4 unique text values
troglitazone: 2 unique text values
tolazamide: 2 unique text values
examide: 1 unique text values
citoglipton: 1 unique text values
insulin: 4 unique text values
glyburide-metformin: 4 unique text values
glipizide-

In [ ]:
# Drop high-missing or low-value columns instantly
cols_to_drop = ['weight', 'payer_code', 'medical_specialty']
X_train = X_train.drop(columns=cols_to_drop, errors='ignore')
X_test = X_test.drop(columns=cols_to_drop, errors='ignore')

In [ ]:
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

In [ ]:
# 1. Define every single negligible, missing, and zero-variance column
all_cols_to_drop = [
    'weight', 'payer_code', 'medical_specialty', # High missing data
    'encounter_id', 'patient_nbr',              # System IDs
    'acetohexamide', 'examide', 'citoglipton', 'metformin-pioglitazone', # Zero variance (1 value)
    'troglitazone', 'tolbutamide', 'tolazamide',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone' # Extremely rare drugs
]

# 2. Drop them safely from both sets
X_train = X_train.drop(columns=all_cols_to_drop, errors='ignore')
X_test = X_test.drop(columns=all_cols_to_drop, errors='ignore')

# 3. Check what text columns are left
remaining_text_cols = X_train.select_dtypes(include=['object']).columns.tolist()
print("Cleaned text columns left to encode:\n", remaining_text_cols)

Cleaned text columns left to encode:
 ['race', 'gender', 'age', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'change', 'diabetesMed']


In [ ]:
# Selects columns where the data type is 'object' (text)
text_columns = X_train.select_dtypes(include=['object']).columns.tolist()

print("Columns with text values:")
print(text_columns)

Columns with text values:
['race', 'gender', 'age', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'change', 'diabetesMed']


In [ ]:
import pandas as pd

def map_icd9_to_category(code):
    if pd.isnull(code) or code == '?':
        return 'Unknown'
    if str(code).startswith(('E', 'V')):
        return 'Other'
    try:
        val = float(code)
        if 390 <= val <= 459 or val == 785:
            return 'Circulatory'
        elif 460 <= val <= 519 or val == 786:
            return 'Respiratory'
        elif 520 <= val <= 579 or val == 787:
            return 'Digestive'
        elif int(val) == 250:
            return 'Diabetes'
        elif 800 <= val <= 999:
            return 'Injury'
        elif 710 <= val <= 739:
            return 'Musculoskeletal'
        elif 580 <= val <= 629 or val == 788:
            return 'Genitourinary'
        else:
            return 'Other'
    except ValueError:
        return 'Other'

# Apply the mapping safely using .loc
for col in ['diag_1', 'diag_2', 'diag_3']:
    X_train.loc[:, col] = X_train[col].apply(map_icd9_to_category)
    X_test.loc[:, col] = X_test[col].apply(map_icd9_to_category)

In [ ]:
# Automatically find all remaining text columns
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# Convert text to binary numbers
X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Align columns so Train and Test are identical shapes
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

print(f"Text encoding complete! New features count: {X_train_encoded.shape[1]}")

Text encoding complete! New features count: 72


In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                'num_medications', 'number_outpatient', 'number_emergency',
                'number_inpatient', 'number_diagnoses']

scaler = StandardScaler()

# Scale train and transform test safely
X_train_encoded[numeric_cols] = scaler.fit_transform(X_train_encoded[numeric_cols])
X_test_encoded[numeric_cols] = scaler.transform(X_test_encoded[numeric_cols])

print("--- ALL PREPROCESSING COMPLETED ---")

--- ALL PREPROCESSING COMPLETED ---


In [ ]:
print("Missing values in Train:", X_train_encoded.isnull().sum().sum())
print("Missing values in Test:", X_test_encoded.isnull().sum().sum())
print("Train Columns match Test Columns perfectly:", list(X_train_encoded.columns) == list(X_test_encoded.columns))
print("Final Dataset Shapes:", X_train_encoded.shape, X_test_encoded.shape)

Missing values in Train: 0
Missing values in Test: 0
Train Columns match Test Columns perfectly: True
Final Dataset Shapes: (81613, 72) (20153, 72)
